> 在线讲义：[第 7 章 · 批判地读 loss 曲线和消融表：一张图先问五个问题](https://wikieden.github.io/math-and-eval-stats/lessons/07-read-curves-and-ablations.html) · 本书《数学与实验统计》

# 第 7 章 · 批判地读 loss 曲线和消融表：一张图先问五个问题

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/07-read-curves-and-ablations.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 种子方差 | 均值、标准差 | `seed_summary(SEEDS_A)` | 71.90、1.33 |
| 第二步 检验 | Welch / Mann-Whitney | `welch(...)`、`mann_whitney_p(...)` | 0.351 / 0.421 |
| 第二步 P(B > A) | 式 (9) | `prob_outperform(SEEDS_A, SEEDS_B)` | 0.68 |
| 第二步 种子数 | 样本量公式 | `seeds_needed(1.31, 1.0)` | 27 |
| 第三步 EMA 滞后 | ln 0.1 / ln β | `lag_after_drop(0.9)`、`lag_formula(0.99)` | 22、230 |

例子全是编的：两个配置各 5 个种子的分数；一条带噪的 loss 曲线做 EMA 平滑；训练 / 验证 loss 的差距。
里程碑 ME-M6（审一张真表）留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind
SEEDS_A = np.array([71.2, 72.8, 70.1, 73.5, 71.9])  # 配置 A 五个种子的准确率（编的）
SEEDS_B = np.array([72.4, 71.0, 73.9, 72.2, 74.1])  # 配置 B（编的）

**函数 `seed_summary`**：均值和种子间标准差（ddof = 1）。

In [ ]:
def seed_summary(x: np.ndarray) -> tuple[float, float]:
    """均值和种子间标准差（ddof = 1）。"""
    return float(x.mean()), float(x.std(ddof=1))

**函数 `welch`**：Welch t 检验（不假设方差相同）的双侧 p 值。

In [ ]:
def welch(a: np.ndarray, b: np.ndarray) -> float:
    """Welch t 检验（不假设方差相同）的双侧 p 值。"""
    return float(ttest_ind(b, a, equal_var=False).pvalue)

**函数 `prob_outperform`**：Bouthillier 2021 式 (9)：P(B > A) = 所有 (b_i, a_j) 配对里 b 赢的比例（平局算一半）。

In [ ]:
def prob_outperform(a: np.ndarray, b: np.ndarray) -> float:
    """Bouthillier 2021 式 (9)：P(B > A) = 所有 (b_i, a_j) 配对里 b 赢的比例（平局算一半）。"""
    wins = (b[:, None] > a[None, :]).sum() + 0.5 * (b[:, None] == a[None, :]).sum()
    return float(wins / (len(a) * len(b)))

**函数 `mann_whitney_p`**：Mann-Whitney U 检验（只看排序）的双侧 p 值。

In [ ]:
def mann_whitney_p(a: np.ndarray, b: np.ndarray) -> float:
    """Mann-Whitney U 检验（只看排序）的双侧 p 值。"""
    return float(mannwhitneyu(b, a, alternative="two-sided").pvalue)

**函数 `ema`**：TensorBoard 式的指数滑动平均：s_t = β·s_{t−1} + (1 − β)·x_t，s_0 = x_0。

In [ ]:
def ema(x: np.ndarray, beta: float) -> np.ndarray:
    """TensorBoard 式的指数滑动平均：s_t = β·s_{t−1} + (1 − β)·x_t，s_0 = x_0。"""
    s = np.empty_like(x)
    s[0] = x[0]
    for t in range(1, len(x)):
        s[t] = beta * s[t - 1] + (1 - beta) * x[t]
    return s

**函数 `loss_curve`**：编一条 loss 曲线：缓慢下降，在 drop_at 步突然降 0.5（比如学习率衰减），每步加噪声（noise = 0 就是干净曲线）。

In [ ]:
def loss_curve(n: int = 600, drop_at: int = 200, seed: int = 0, noise: float = 0.15) -> np.ndarray:
    """编一条 loss 曲线：缓慢下降，在 drop_at 步突然降 0.5（比如学习率衰减），每步加噪声（noise = 0 就是干净曲线）。"""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    base = 2.0 * np.exp(-t / 300) + 0.5 - 0.5 * (t >= drop_at)
    return base + rng.normal(0, noise, n)

**函数 `lag_after_drop`**：在干净的阶跃上量：平滑后的曲线要过多少步，才走完这次 0.5 下降的 frac（只看阶跃，扣掉曲线本身的缓降）。

In [ ]:
def lag_after_drop(beta: float, drop_at: int = 200, frac: float = 0.9) -> int:
    """在干净的阶跃上量：平滑后的曲线要过多少步，才走完这次 0.5 下降的 frac（只看阶跃，扣掉曲线本身的缓降）。"""
    x = loss_curve(noise=0.0, drop_at=drop_at)
    ref = loss_curve(noise=0.0, drop_at=10**9)[: len(x)]  # 同一条曲线但不突降
    gap = ema(ref, beta) - ema(x, beta)                     # 平滑后已经"感受到"的下降
    for k in range(drop_at, len(x)):
        if gap[k] >= frac * 0.5:
            return k - drop_at + 1
    return -1

**函数 `lag_formula`**：阶跃输入下 EMA 走完 frac 要的步数：1 − β^k ≥ frac → k = ⌈ln(1 − frac)/ln β⌉。

In [ ]:
def lag_formula(beta: float, frac: float = 0.9) -> int:
    """阶跃输入下 EMA 走完 frac 要的步数：1 − β^k ≥ frac → k = ⌈ln(1 − frac)/ln β⌉。"""
    return int(np.ceil(np.log(1 - frac) / np.log(beta))) if beta > 0 else 1

**函数 `seeds_needed`**：要分清相差 delta 的两个配置，每边要几个种子：n ≈ 2·z²·sd²/δ²（z = z_α/2 + z_β ≈ 1.96 + 0.84）。

In [ ]:
def seeds_needed(sd: float, delta: float, z: float = 2.8) -> int:
    """要分清相差 delta 的两个配置，每边要几个种子：n ≈ 2·z²·sd²/δ²（z = z_α/2 + z_β ≈ 1.96 + 0.84）。"""
    return int(np.ceil(2 * z**2 * sd**2 / delta**2))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 种子方差：同一个配置换种子，结果差多少")
for name, x in (("A", SEEDS_A), ("B", SEEDS_B)):
    m, sd = seed_summary(x)
    print(f"  {name}：均值 {m:.2f}，种子间标准差 {sd:.2f}，最好的一个 {x.max():.1f}")
print(f"  B − A = {SEEDS_B.mean() - SEEDS_A.mean():+.2f}；只挑最好的种子比：{SEEDS_B.max() - SEEDS_A.max():+.1f}")

In [ ]:
print("\n第二步 差异是不是噪声")
print(f"  Welch t 检验 p = {welch(SEEDS_A, SEEDS_B):.3f}；Mann-Whitney p = {mann_whitney_p(SEEDS_A, SEEDS_B):.3f}")
print(f"  P(B > A)（随便挑一个 B 种子赢随便一个 A 种子的概率）= {prob_outperform(SEEDS_A, SEEDS_B):.2f}")
sd = np.sqrt((SEEDS_A.var(ddof=1) + SEEDS_B.var(ddof=1)) / 2)
print(f"  合并标准差 {sd:.2f}：分清 1 个点的差要每边 {seeds_needed(sd, 1.0)} 个种子")

In [ ]:
print("\n第三步 平滑会拖后腿：loss 在第 200 步突降 0.5")
noisy = loss_curve()
print(f"  原始曲线每步噪声标准差 0.15；β = 0.9 平滑后残余抖动 {np.std(ema(noisy, 0.9)[300:] - loss_curve(noise=0)[300:]):.3f}")
for beta in (0.0, 0.9, 0.99):
    print(f"  EMA β = {beta}：走完 90% 的下降要 {lag_after_drop(beta)} 步（公式 {lag_formula(beta)}）")

In [ ]:
print("\n第四步 过拟合：看训练和验证的差距，不看单条曲线")
train = np.array([2.1, 1.4, 1.0, 0.7, 0.5, 0.35, 0.25])
val = np.array([2.2, 1.6, 1.3, 1.2, 1.25, 1.35, 1.5])
print(f"  验证 loss 最低在第 {int(val.argmin())} 个检查点；之后训练还在降，验证在升（差距 {val[-1] - train[-1]:.2f}）")

## 3. 代码里才看得到的

1. **广播做配对比较**：`b[:, None] > a[None, :]` 一次比出 5 × 5 = 25 对，不用双重循环。
2. **滞后要在干净的阶跃上量**：带噪声的曲线上，平滑值什么时候"越过 90%"会被噪声左右；代码用同一条曲线的无噪声版本、再扣掉不突降的版本，得到的步数和公式完全一致。
3. **TensorBoard 的平滑有去偏**：实际实现会除以 $1 - \beta^t$ 修正开头那段被 $s_0$ 拉低的值，开头的形状和本章的简单版不同；稳定以后两者一样。

## 4. 自己改一改

1. 把 `SEEDS_B` 全部加 2，看 Welch p 值和 P(B > A) 各变成多少。
2. 把 `loss_curve` 的噪声从 0.15 改成 0.5，β = 0.9 的平滑曲线还看得清那次突降吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先问换种子差多少，差值小于标准差就别下结论；
> 均值差不如赢的概率，几十个种子才分得清一个点；
> 平滑像看后视镜，β 越大拖得越久；
> 训练降了看验证，五个问题逐个问。